# Big Cypress

## Setup

In [ ]:
# Import modules
import pathlib
import urllib.request
import pyvista as pv
import pyvista_cad
import pymeshfix as mf

# Set mode
pv.set_jupyter_backend("static")

# Set paths
file = pathlib.Path("big-cypress.ply")
root = pathlib.Path.cwd().parent
source = root / "clouds"
source.mkdir(parents=True, exist_ok=True)
prints = root / "prints"
prints.mkdir(parents=True, exist_ok=True)
images = root / "images"
data = source / file

## Download

In [ ]:
# Download dataset
url = "https://osf.io/wjg36/download"
try:
    urllib.request.urlretrieve(url, data)
except Exception as e:
    print(f"Error downloading file: {e}")

## Read

In [ ]:
# Read point cloud
cloud = pv.read(data)

# Plot point cloud
res = 2
dimensions = 1000 * res
size = 4 * res
screenshot = images / f"{file.stem}-01.png"
pl = pv.Plotter(
    off_screen=True,
    window_size=(dimensions, dimensions)
    )
pl.enable_eye_dome_lighting()
pl.add_points(
    cloud,
    rgb=True,
    render_points_as_spheres=True,
    point_size=size,
    ambient=0.7,
    diffuse=0.6,
    )
pl.view_yz(negative=True)
pl.camera.zoom(1.45)
pl.show(screenshot=screenshot)

## Splat

In [ ]:
# Splat point cloud
dimensions = (500, 500, 500)
volume = cloud.gaussian_splatting(
    radius=0.004,
    dimensions=dimensions,
    progress_bar=True
    )
volume = volume.threshold(0.01)

# Extract mesh
mesh = volume.extract_surface(algorithm=None)

# Smooth mesh
mesh = mesh.smooth_taubin(n_iter=50, pass_band=0.1)

# Scale mesh
factor = [50, 50, 50]
mesh = mesh.scale(factor)

# Clean mesh
mesh = mesh.clean()
mesh = mf.MeshFix(mesh)
mesh = mesh.mesh

# Plot mesh
res = 2
dimensions = 1000 * res
screenshot = images / f"{file.stem}-02.png"
pl = pv.Plotter(
    off_screen=True,
    window_size=(dimensions, dimensions)
    )
pl.enable_ssao()
pl.add_mesh(mesh, color="white")
pl.view_yz(negative=True)
pl.camera.zoom(1.45)
pl.show(screenshot=screenshot)

## Export

In [ ]:
# Export mesh
export = prints / file.with_suffix(".3mf")
mesh.cad.to_3mf(export, units="mm")